# Notebook 2 — Régression

## Prédiction du prix d'un trajet de taxi

**Objectif :** comparer plusieurs modèles de régression et déployer uniquement le plus performant.

### 1. Importation et lecture des données

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
BASE_DIR = Path.cwd()
ARTIFACTS_DIR = BASE_DIR / 'streamlit_app' / 'artifacts'
df = pd.read_csv(BASE_DIR / 'taxi_trip_pricing.csv')
df.head()

### 2. Nettoyage et préparation
La cible est `Trip_Price`, une valeur numérique continue : c'est donc un problème de régression. Les valeurs manquantes sont traitées dans le pipeline afin d'éviter les fuites de données.

In [ ]:
target = 'Trip_Price'
df = df.dropna(subset=[target]).copy()
X = df.drop(columns=target)
y = df[target]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
print('Variables numériques :', numeric_features)
print('Variables catégorielles :', categorical_features)

### 3. Entraînement et comparaison
Les quatre modèles sont conservés dans le notebook. La sélection du meilleur modèle se fait avec le **RMSE** : plus il est faible, meilleur est le modèle.

In [ ]:
models = {
    'Linear Regression': (LinearRegression(), {}),
    'Ridge': (Ridge(), {'model__alpha': [0.1, 1, 10, 100]}),
    'Decision Tree': (DecisionTreeRegressor(random_state=RANDOM_STATE), {'model__max_depth': [None, 4, 6, 8, 10], 'model__min_samples_leaf': [1, 2, 4]}),
    'Random Forest': (RandomForestRegressor(random_state=RANDOM_STATE), {'model__n_estimators': [100, 200], 'model__max_depth': [None, 6, 10], 'model__min_samples_leaf': [1, 2, 4]})
}

fitted_models, rows = {}, []
for name, (model, params) in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    grid = GridSearchCV(pipeline, params, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    prediction = grid.predict(X_test)
    fitted_models[name] = grid.best_estimator_
    rows.append({
        'Model': name,
        'R2': r2_score(y_test, prediction),
        'MAE': mean_absolute_error(y_test, prediction),
        'RMSE': mean_squared_error(y_test, prediction) ** 0.5,
        'Best parameters': grid.best_params_
    })

results = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
results

### 4. Choix et sauvegarde du meilleur modèle
Le pipeline complet est enregistré : il contient le prétraitement et le modèle. L'application peut donc prédire avec de nouvelles données sans reconstruire manuellement les encodeurs et le scaler.

In [ ]:
best_model_name = results.loc[0, 'Model']
best_regressor = fitted_models[best_model_name]
joblib.dump(best_regressor, ARTIFACTS_DIR / 'best_regressor.joblib')
print(f'Modèle retenu : {best_model_name}')

### Conclusion
Le meilleur modèle selon le RMSE est déployé dans l'onglet **Régression** de l'application Streamlit. Les autres modèles restent disponibles dans ce notebook.